In [ ]:
# ruff: noqa
!git clone https://github.com/dramirezbe/DataBase-RF-FM-88MHz-108MHz-Bogota-Funza.git

In [ ]:
import ast
import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

csv_paths = glob.glob("DataBase-RF-FM-88MHz-108MHz-Bogota-Funza/Node*.csv")
node_frames = {}

# Load every node CSV into memory, skipping the known outlier file Node8.
for csv_path in tqdm(csv_paths, desc="Loading CSV files"):
    if "Node8" in csv_path:
        continue

    node_name = csv_path.replace(".csv", "")
    node_frames[node_name] = pd.read_csv(csv_path)

In [ ]:
row_index_to_plot = 103  # Change this index to inspect a different row [0, 22]

plt.figure(figsize=(21, 12))

for node_name, frame in node_frames.items():
    # Parse the PSD list for the selected row and build the frequency axis [MHz].
    pxx = ast.literal_eval(frame["pxx"].iloc[row_index_to_plot])
    frequency_axis_mhz = np.linspace(88, 108, len(pxx))

    plt.plot(frequency_axis_mhz, pxx, label=node_name, linewidth=1, alpha=0.8)

plt.title(f"FM Spectrum Comparison (Row {row_index_to_plot})")
plt.xlabel("Frequency [MHz]")
plt.ylabel("Power [dBm]")
plt.xlim(min(frequency_axis_mhz), max(frequency_axis_mhz) - 1)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
selected_row_index = 0  # Change this index to inspect a different row [0, 105]

# Container: {node_name: {row_index: pxx_array}}
indexed_pxx = {}

plt.figure(figsize=(21, 12))

for node_name, frame in node_frames.items():
    if node_name not in indexed_pxx:
        indexed_pxx[node_name] = {}

    pxx_raw = frame["pxx"].iloc[selected_row_index]
    pxx = np.array(ast.literal_eval(pxx_raw))
    indexed_pxx[node_name][selected_row_index] = pxx

    frequency_axis_mhz = np.linspace(88, 108, len(pxx))
    plt.plot(frequency_axis_mhz, pxx, label=node_name, alpha=0.7)

plt.xlabel("Frequency [MHz]")
plt.ylabel("Power Spectral Density")
plt.title(f"PSD Comparison - Row {selected_row_index}")
plt.legend(loc="best", fontsize="small")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# indexed_pxx is now ready for later use.
# Example: indexed_pxx["Node_A"][5]

In [ ]:
selected_row_index = 22  # Change this index to inspect a different row [0, 105]

# Container: {node_name: {row_index: pxx_array}}
indexed_pxx = {}
noise_floor_estimates_db = {}

plt.figure(figsize=(21, 12))

for node_name, frame in node_frames.items():
    if node_name not in indexed_pxx:
        indexed_pxx[node_name] = {}

    pxx_raw = frame["pxx"].iloc[selected_row_index]
    pxx = np.array(ast.literal_eval(pxx_raw))
    indexed_pxx[node_name][selected_row_index] = pxx

    frequency_axis_mhz = np.linspace(88, 108, len(pxx))

    # Estimate the noise floor with the histogram mode, which is robust on multimodal PSDs.
    counts, bins = np.histogram(pxx, bins=50)
    noise_floor_db = bins[np.argmax(counts)]
    noise_floor_estimates_db[node_name] = noise_floor_db

    plt.plot(frequency_axis_mhz, pxx, label=node_name, alpha=0.7)
    plt.axhline(y=noise_floor_db, linestyle="--", alpha=0.3)

plt.xlabel("Frequency [MHz]")
plt.ylabel("Power Spectral Density [dB]")
plt.title(f"PSD Comparison - Row {selected_row_index}")
plt.legend(loc="best", fontsize="small")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nNoise floor estimates (row {selected_row_index}):")
print("-" * 50)

ordered_node_names = sorted(noise_floor_estimates_db.keys())
noise_floor_array_db = np.array(
    [noise_floor_estimates_db[node_name] for node_name in ordered_node_names]
)

print("Node order:", ordered_node_names)
print("Noise floors [dB]:", noise_floor_array_db)
print(
    f"Statistics -> mean: {np.mean(noise_floor_array_db):.2f} dB, "
    f"std: {np.std(noise_floor_array_db):.2f} dB, "
    f"min: {np.min(noise_floor_array_db):.2f} dB, "
    f"max: {np.max(noise_floor_array_db):.2f} dB"
)

print("\nDetailed table:")
print(f"{'Node':<20} {'Noise Floor [dB]':>15}")
print("-" * 35)
for node_name in ordered_node_names:
    print(f"{node_name:<20} {noise_floor_estimates_db[node_name]:>15.2f}")

In [ ]:
selected_row_index = 22  # Change this index to inspect a different row [0, 105]

indexed_pxx = {}
noise_floor_estimates_db = {}

# =============================================================================
# PASS 1: Estimate the noise floor for each node with the histogram-mode method.
# =============================================================================
for node_name, frame in node_frames.items():
    if node_name not in indexed_pxx:
        indexed_pxx[node_name] = {}

    pxx_raw = frame["pxx"].iloc[selected_row_index]
    pxx = np.array(ast.literal_eval(pxx_raw))
    indexed_pxx[node_name][selected_row_index] = pxx

    counts, bins = np.histogram(pxx, bins=50)
    noise_floor_db = bins[np.argmax(counts)]
    noise_floor_estimates_db[node_name] = noise_floor_db

# =============================================================================
# Compute the global reference noise floor as the mean of node estimates.
# =============================================================================
noise_floor_array_db = np.array(list(noise_floor_estimates_db.values()))
global_noise_floor_mean_db = np.mean(noise_floor_array_db)

print(f"\nNoise floor alignment (row {selected_row_index})")
print("-" * 60)
print(f"Global reference noise floor (mean): {global_noise_floor_mean_db:.2f} dB")
print("Per-node offsets to apply: {node: offset_dB}")
for node_name, node_noise_floor_db in noise_floor_estimates_db.items():
    offset_db = global_noise_floor_mean_db - node_noise_floor_db
    print(f"  {node_name:<15} -> offset = {offset_db:+.2f} dB")

# =============================================================================
# PASS 2: Plot the original PSD traces for comparison.
# =============================================================================
frequency_axis_mhz = np.linspace(
    88, 108, len(indexed_pxx[node_name][selected_row_index])
)

plt.figure(figsize=(21, 8))
for node_name in node_frames:
    pxx = indexed_pxx[node_name][selected_row_index]
    plt.plot(frequency_axis_mhz, pxx, label=node_name, alpha=0.6, linewidth=0.8)

plt.xlabel("Frequency [MHz]")
plt.ylabel("Power Spectral Density [dB] (Original)")
plt.title(f"Original PSD Comparison - Row {selected_row_index}")
plt.legend(loc="best", fontsize="x-small", ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# =============================================================================
# PASS 3: Plot recentered PSD traces aligned to the common noise floor.
# =============================================================================
plt.figure(figsize=(21, 8))

recentered_pxx_by_node = {}
for node_name in node_frames:
    pxx_original = indexed_pxx[node_name][selected_row_index]
    node_noise_floor_db = noise_floor_estimates_db[node_name]

    # Shift each PSD so that every node matches the same global noise floor.
    offset_db = global_noise_floor_mean_db - node_noise_floor_db
    pxx_recentered = pxx_original + offset_db
    recentered_pxx_by_node[node_name] = pxx_recentered

    plt.plot(
        frequency_axis_mhz, pxx_recentered, label=node_name, alpha=0.7, linewidth=1.0
    )

plt.axhline(
    y=global_noise_floor_mean_db,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=f"Global noise floor ({global_noise_floor_mean_db:.2f} dB)",
    zorder=5,
)

plt.xlabel("Frequency [MHz]")
plt.ylabel("Power Spectral Density [dB] (Recentered)")
plt.title(
    f"Recentered PSD Comparison - Row {selected_row_index}\n"
    "(All nodes aligned to a common noise floor)"
)
plt.legend(loc="best", fontsize="x-small", ncol=2)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nRecentered data stored in 'recentered_pxx_by_node'.")
print(
    f"Example: recentered_pxx_by_node['{list(node_frames.keys())[0]}'] -> aligned PSD array"
)

In [ ]:
from scipy.stats import pearsonr

print(
    f"Available row indices: {list(node_frames.values())[0].index.min()} to {list(node_frames.values())[0].index.max()}"
)

selected_row_index = 22  # Change this index to inspect a different row [1, 103]

indexed_pxx = {}
noise_floor_estimates_db = {}

# =============================================================================
# PASS 1: Estimate node noise floors and build recentered, normalized PSDs.
# =============================================================================
for node_name, frame in node_frames.items():
    if node_name not in indexed_pxx:
        indexed_pxx[node_name] = {}

    pxx_raw = frame["pxx"].iloc[selected_row_index]
    pxx = np.array(ast.literal_eval(pxx_raw))
    indexed_pxx[node_name][selected_row_index] = pxx

    counts, bins = np.histogram(pxx, bins=50)
    noise_floor_db = bins[np.argmax(counts)]
    noise_floor_estimates_db[node_name] = noise_floor_db

noise_floor_array_db = np.array(list(noise_floor_estimates_db.values()))
global_noise_floor_mean_db = np.mean(noise_floor_array_db)

normalized_pxx_by_node = {}
for node_name in node_frames:
    pxx_original = indexed_pxx[node_name][selected_row_index]
    offset_db = global_noise_floor_mean_db - noise_floor_estimates_db[node_name]
    pxx_recentered = pxx_original + offset_db

    # Z-normalize each recentered PSD for a fair correlation comparison.
    pxx_normalized = (pxx_recentered - np.mean(pxx_recentered)) / (
        np.std(pxx_recentered) + 1e-8
    )
    normalized_pxx_by_node[node_name] = pxx_normalized

# =============================================================================
# PASS 2: Compute the pairwise Pearson correlation matrix.
# =============================================================================
node_names = sorted(node_frames.keys())
node_count = len(node_names)
correlation_matrix = np.zeros((node_count, node_count))

print(f"\nComputing pairwise correlations (row {selected_row_index})...")
for row_index, node_name_i in enumerate(node_names):
    for column_index, node_name_j in enumerate(node_names):
        if row_index == column_index:
            correlation_matrix[row_index, column_index] = 1.0
        elif column_index > row_index:
            correlation_value, _ = pearsonr(
                normalized_pxx_by_node[node_name_i],
                normalized_pxx_by_node[node_name_j],
            )
            correlation_matrix[row_index, column_index] = correlation_value
            correlation_matrix[column_index, row_index] = correlation_value

# =============================================================================
# PASS 3: Rank nodes by cumulative correlation score.
# =============================================================================
cumulative_scores = np.sum(correlation_matrix, axis=1) - 1.0
ranking_indices = np.argsort(cumulative_scores)[::-1]
ranked_node_names = [node_names[index] for index in ranking_indices]
ranked_scores = cumulative_scores[ranking_indices]
ranked_correlation_matrix = correlation_matrix[np.ix_(ranking_indices, ranking_indices)]

print(f"\nCumulative correlation scores (row {selected_row_index})")
print("-" * 70)
print(f"{'Rank':<6} {'Node':<20} {'Cumulative Score':>18} {'Avg Corr':>12}")
print("-" * 70)
for rank, (node_name, score) in enumerate(
    zip(ranked_node_names, ranked_scores), start=1
):
    average_correlation = score / (node_count - 1)
    print(f"{rank:<6} {node_name:<20} {score:>18.4f} {average_correlation:>12.4f}")

# =============================================================================
# PASS 4: Identify the most representative and most distinctive nodes.
# =============================================================================
most_representative_index = ranking_indices[0]
most_distinctive_index = ranking_indices[-1]

most_representative_node = node_names[most_representative_index]
most_distinctive_node = node_names[most_distinctive_index]

print("\nKey nodes:")
print(
    f"   Highest cumulative correlation: {most_representative_node} ({ranked_scores[0]:.4f})"
)
print(
    f"   Lowest cumulative correlation: {most_distinctive_node} ({ranked_scores[-1]:.4f})"
)
print(
    "   These nodes approximate the most typical and most distinctive spectral profiles."
)

# =============================================================================
# PASS 5: Plot the top-ranked and bottom-ranked nodes after recentering.
# =============================================================================
frequency_axis_mhz = np.linspace(88, 108, len(normalized_pxx_by_node[node_names[0]]))

plt.figure(figsize=(14, 7))

representative_pxx = indexed_pxx[most_representative_node][selected_row_index]
representative_offset_db = (
    global_noise_floor_mean_db - noise_floor_estimates_db[most_representative_node]
)
representative_pxx_recentered = representative_pxx + representative_offset_db
plt.plot(
    frequency_axis_mhz,
    representative_pxx_recentered,
    label=f"{most_representative_node} (Highest: {ranked_scores[0]:.3f})",
    color="green",
    linewidth=2,
    alpha=0.9,
)

distinctive_pxx = indexed_pxx[most_distinctive_node][selected_row_index]
distinctive_offset_db = (
    global_noise_floor_mean_db - noise_floor_estimates_db[most_distinctive_node]
)
distinctive_pxx_recentered = distinctive_pxx + distinctive_offset_db
plt.plot(
    frequency_axis_mhz,
    distinctive_pxx_recentered,
    label=f"{most_distinctive_node} (Lowest: {ranked_scores[-1]:.3f})",
    color="red",
    linewidth=2,
    alpha=0.9,
    linestyle="--",
)

plt.axhline(
    y=global_noise_floor_mean_db,
    color="gray",
    linestyle=":",
    linewidth=1,
    alpha=0.5,
    label=f"Global noise floor ({global_noise_floor_mean_db:.2f} dB)",
)

plt.xlabel("Frequency [MHz]", fontsize=12)
plt.ylabel("Power Spectral Density [dB] (Recentered)", fontsize=12)
plt.title(
    f"Most vs Least Correlated Nodes - Row {selected_row_index}\n"
    "(Based on cumulative pairwise correlation)",
    fontsize=14,
    fontweight="bold",
)
plt.legend(loc="best", fontsize=10)
plt.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()

# =============================================================================
# PASS 6: Visualize the reordered correlation matrix.
# =============================================================================
plt.figure(figsize=(10, 8))
image = plt.imshow(
    ranked_correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto"
)

plt.xticks(
    np.arange(node_count),
    [name[:12] + ".." if len(name) > 14 else name for name in ranked_node_names],
    rotation=45,
    ha="right",
    fontsize=8,
)
plt.yticks(
    np.arange(node_count),
    [name[:12] + ".." if len(name) > 14 else name for name in ranked_node_names],
    fontsize=8,
)

plt.xlabel("Nodes (sorted by descending cumulative correlation)", fontsize=11)
plt.ylabel("Nodes (sorted by descending cumulative correlation)", fontsize=11)
plt.title(
    f"Reordered Correlation Matrix - Row {selected_row_index}\n"
    "(Top-left: the most mutually correlated cluster)",
    fontsize=13,
    pad=20,
)
plt.colorbar(image, label="Pearson correlation coefficient")
plt.grid(False)
plt.tight_layout()
plt.show()

# =============================================================================
# Output: Save results for downstream use.
# =============================================================================
correlation_results = {
    "row_plot": selected_row_index,
    "node_names": node_names,
    "corr_matrix": correlation_matrix,
    "cumulative_scores": dict(zip(node_names, cumulative_scores)),
    "ranking": ranked_node_names,
    "ranked_scores": dict(zip(ranked_node_names, ranked_scores)),
    "most_representative": most_representative_node,
    "most_distinctive": most_distinctive_node,
    "global_noise_mean": global_noise_floor_mean_db,
}

print("\nCorrelation analysis complete.")
print("   Results stored in the 'correlation_results' dictionary.")
print("   Use 'correlation_results[\"most_representative\"]' for prototype selection.")
print("   Use 'correlation_results[\"most_distinctive\"]' for anomaly detection.")